# Classwork: The Analyst's Toolkit

**Names:** Joshua Brissette & _(partner)_

After learning about different tools (or a toolkit-- Descriptive statistics, Correlation, Regression/Multiple regression, Chi-square test) of working with data in the previous Mods, in this acitivity you will apply your real skills on the data you have.

  Suppose your company hands you `customers.csv` (40 customers: `age`, `income`, `spending`, `region`, `subscribed`) and a list of questions. You are the analyst. Answer each one with the right tool/technique you decide to apply.

Work in pairs. Write answers in the markdown cells provided.

## Setup
Import the libraries you'll need and load the data.

In [1]:
# import pandas (and numpy, scipy.stats, statsmodels.api as you need them)
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
# load customers.csv into a DataFrame
customers = pd.read_csv('customers.csv')
# take a first look (head, info)
customers.head()


,customer_id,age,income,spending,region,subscribed
0,1,25,61.3,39.1,West,No
1,2,25,54.9,42.1,South,Yes
2,3,51,105.1,59.9,South,No
3,4,39,NaN,49.9,South,No
4,5,43,72.4,48.3,East,No


## Part 0: Know your data, then clean it

This data is dirty on purpose. Find and fix **three** problems before you analyze anything.

In [2]:
print("Missing values before cleaning:")
print(customers.isna().sum())
print("\nRegion labels before cleaning:", customers['region'].unique())
print("Income range before cleaning:", customers['income'].min(), "to", customers['income'].max())

# Problem 3: an income of -50 is impossible -> mark it missing first
customers.loc[customers['income'] < 0, 'income'] = np.nan

# Problem 1: fill missing income with the median (robust to outliers)
income_median = customers['income'].median()
customers['income'] = customers['income'].fillna(income_median)

# Problem 2: 'east' should be 'East' -> standardize the case
customers['region'] = customers['region'].str.strip().str.title()

print("\nMissing values after cleaning:", customers.isna().sum().sum())
print("Region labels after cleaning:", customers['region'].unique())
print(f"Income median used for filling: {income_median}")
print("Income range after cleaning:", customers['income'].min(), "to", customers['income'].max())

Missing values before cleaning:
customer_id    0
age            0
income         2
spending       0
region         0
subscribed     0
dtype: int64

Region labels before cleaning: <StringArray>
['West', 'South', 'East', 'east']
Length: 4, dtype: str
Income range before cleaning: -50.0 to 114.5

Missing values after cleaning: 0
Region labels after cleaning: <StringArray>
['West', 'South', 'East']
Length: 3, dtype: str
Income median used for filling: 84.7
Income range after cleaning: 53.3 to 114.5


**Explain each issue:** what you found and how you handled it.

*Your answer:*
1. **Missing values:** `income` had 2 missing values (NaN). I filled them with the **median** income (84.7) instead of the mean, because the median isn't pulled around by extreme or bad values (like the −50 below).
2. **Inconsistent category label:** `region` had one value written as `east` instead of `East`, which would have created a fake 4th region. I standardized the labels with `.str.title()`, giving 3 regions: East, West, South.
3. **Impossible value:** one customer had an income of **−50**, which is impossible. I first set it to NaN (so it wouldn't distort the median), and then it was filled with the median along with the other missing values.

## Part 1: Pick the tool for each question

For each question: (a) name the data types involved, (b) name the tool, (c) run it, (d) answer in one plain sentence.

### Q1. Do customers who earn more also spend more?

In [3]:
# data types: income = continuous (numeric), spending = continuous (numeric)
# tool: Pearson correlation
r, p = stats.pearsonr(customers['income'], customers['spending'])
print(f"Pearson r = {r:.3f}, p-value = {p:.2e}")

Pearson r = 0.717, p-value = 1.92e-07


*Your answer:* (a) income and spending are both continuous numeric variables; (b) tool: **Pearson correlation**; (c) r = 0.717, p = 1.9 × 10⁻⁷; (d) **Yes.** There is a strong, statistically significant positive relationship: customers who earn more tend to spend more.

### Q2. Is a customer's region related to whether they subscribed?

In [4]:
# data types: region = categorical (nominal), subscribed = categorical (Yes/No)
# tool: Chi-square test of independence
table = pd.crosstab(customers['region'], customers['subscribed'])
print(table)
chi2, p, dof, expected = stats.chi2_contingency(table)
print(f"\nChi-square = {chi2:.3f}, dof = {dof}, p-value = {p:.4f}")

subscribed  No  Yes
region             
East         5   16
South        6    3
West         9    1

Chi-square = 13.162, dof = 2, p-value = 0.0014


*Your answer:* (a) region and subscribed are both categorical; (b) tool: **Chi-square test of independence** on a `pd.crosstab` contingency table; (c) χ² = 13.16, df = 2, p = 0.0014; (d) **Yes.** Region and subscription are significantly related (p < 0.05). East customers mostly subscribed (16 of 21), while West (1 of 10) and South (3 of 9) mostly did not.

### Q3. Can we predict spending from income and age together? How much do they explain? (You can review your Assignment 4 for this)

In [5]:
# data types: predicting a continuous outcome (spending) from TWO numeric predictors (income, age)
# tool: Multiple linear regression (OLS)
X = sm.add_constant(customers[['income', 'age']])
y = customers['spending']
model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               spending   R-squared:                       0.520
Model:                            OLS   Adj. R-squared:                  0.494
Method:                 Least Squares   F-statistic:                     20.04
Date:                Mon, 09 Nov 2026   Prob (F-statistic):           1.27e-06
Time:                        13:40:07   Log-Likelihood:                -131.11
No. Observations:                  40   AIC:                             268.2
Df Residuals:                      37   BIC:                             273.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         19.2622      6.000      3.210      0.0

*Your answer:* (a) one continuous outcome (spending) with two numeric predictors (income, age); (b) tool: **Multiple linear regression (OLS)**; (c) R² = 0.520, F = 20.04, p = 1.27 × 10⁻⁶; income coefficient = 0.359 (p = 0.003), age coefficient = 0.103 (p = 0.515); (d) **Together, income and age explain about 52% of the variation in spending, and the model is significant. But only income is a significant predictor; age adds almost nothing (p = 0.515 > 0.05).**

### Q4. Which region has the most customers, and what is the average income overall?

In [6]:
# Descriptive statistics
print("Customers per region:")
print(customers['region'].value_counts())
print(f"\nAverage income overall: {customers['income'].mean():.2f}")

Customers per region:
region
East     21
West     10
South     9
Name: count, dtype: int64

Average income overall: 86.46


*Your answer:* (a) region is categorical, income is continuous; (b) tool: **Descriptive statistics** (a count for the category, a mean for the number); (d) **East** has the most customers (21 of 40), and the average income overall is **86.46** (after cleaning).

## Part 2: The reveal

Answer these two in a sentence each (no code needed).

**a)** Look back at your four tools (Correlation, Chi-square test, Multiple regression, & Descriptive statistic)). What decided which tool you used each time? (Hint: it was not the wording of the question.)

*Your answer:* The **data types** of the variables involved (and how many there were) decided the tool: two continuous variables → correlation; two categorical variables → chi-square; a continuous outcome with several predictors → multiple regression; summarizing a single variable → descriptive statistics.


**b)** In Q1 you found a relationship. Can you tell the company that higher income *causes* higher spending? Why or why not?

*Your answer:* **No.** Correlation shows that income and spending move together, not that one causes the other. This is observational data, so a third factor (such as age, job, or lifestyle) could drive both, and only a controlled experiment could show causation.

## Bonus

In Q2 you ran a chi-square test. Look at its **expected counts** (not the observed counts). Do they all meet the assumption you learned in the chi-square module? What would you tell the company about how much to trust that result?

In [7]:
expected_df = pd.DataFrame(expected, index=table.index, columns=table.columns)
print("Expected counts:")
print(expected_df.round(2))
print("\nAll expected counts >= 5?", (expected_df >= 5).all().all())
print("Cells with expected count < 5:", (expected_df < 5).sum().sum(), "of", expected_df.size)

Expected counts:
subscribed    No   Yes
region                
East        10.5  10.5
South        4.5   4.5
West         5.0   5.0

All expected counts >= 5? False
Cells with expected count < 5: 2 of 6


*Your answer:* **No, not all expected counts meet the assumption.** The chi-square test assumes every expected count is at least 5, but South has expected counts of **4.5** (No) and **4.5** (Yes), so 2 of the 6 cells (33%) fall below 5. West is exactly 5.0. With only 40 customers, the p-value (0.0014) is an approximation that may not be reliable. I would tell the company the result **suggests** that region and subscription are related, but it should be treated with caution. They should collect more data (especially from South and West) or confirm it with an exact test such as Fisher's exact test (or by combining South and West) before making decisions based on it.

### Push your code to GitHub